### What is Reflection?

Reflection is a prompting strategy aimed at enhancing the quality and accuracy of outputs generated by AI agents. It involves getting the agent to pause, review, and critique its own outputs before finalizing them. This iterative process helps in reducing errors and improving performance over time.

In [40]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x177ed4b90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x177ebb850>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [41]:
from typing import List, Sequence
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

generation_prompt=ChatPromptTemplate.from_messages(
[
        (
            "system",
            "You are a professional LinkedIn content assistant tasked with crafting engaging, insightful, and well-structured LinkedIn posts."
            " Generate the best LinkedIn post possible for the user's request."
            " If the user provides feedback or critique, respond with a refined version of your previous attempts, improving clarity, tone, or engagement as needed.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [42]:
generate_chain=generation_prompt | model

In [43]:
reflection_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a professional LinkedIn content strategist and thought leadership expert. Your task is to critically evaluate the given LinkedIn post and provide a comprehensive critique. Follow these guidelines:

        1. Assess the post’s overall quality, professionalism, and alignment with LinkedIn best practices.
        2. Evaluate the structure, tone, clarity, and readability of the post.
        3. Analyze the post’s potential for engagement (likes, comments, shares) and its effectiveness in building professional credibility.
        4. Consider the post’s relevance to the author’s industry, audience, or current trends.
        5. Examine the use of formatting (e.g., line breaks, bullet points), hashtags, mentions, and media (if any).
        6. Evaluate the effectiveness of any call-to-action or takeaway.

        Provide a detailed critique that includes:
        - A brief explanation of the post’s strengths and weaknesses.
        - Specific areas that could be improved.
        - Actionable suggestions for enhancing clarity, engagement, and professionalism.

        Your critique will be used to improve the post in the next revision step, so ensure your feedback is thoughtful, constructive, and practical.
        """
    ),
    MessagesPlaceholder(variable_name="messages")
])

In [44]:
reflection_chain=reflection_prompt | model

In [45]:
from langgraph.graph import MessageGraph
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import END, MessageGraph, StateGraph

In [46]:
graph = MessageGraph()


/var/folders/qz/mw466yp17txddtx0y1qsb1q00000gn/T/ipykernel_2858/1161755010.py:1: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = MessageGraph()


In [47]:
def generation_node(state:Sequence[BaseMessage])->List[BaseMessage]:
    generated_post=generate_chain.invoke({"messages":state})
    return [AIMessage(content=generated_post.content)]

In [48]:
def reflection_node(messages: Sequence[BaseMessage]) -> List[BaseMessage]:
    res = reflection_chain.invoke({"messages": messages})  # Passes messages as input to reflect_chain
    return [HumanMessage(content=res.content)]

In [49]:
graph.add_node("generate", generation_node)

In [50]:
graph.add_node("reflect", reflection_node)


In [51]:
graph.add_edge("reflect", "generate")

In [52]:
graph.set_entry_point("generate")

In [53]:
def should_continue(state: List[BaseMessage]):
    print(state)
    print(len(state))
    print("----------------------------------------------------------------------")
    if len(state) > 6:
        return END
    return "reflect"

In [54]:
graph.add_conditional_edges("generate", should_continue)

In [55]:
workflow = graph.compile()

In [56]:
inputs = HumanMessage(content="""Write a linkedin post on getting a software developer job at IBM under 160 characters""")

In [57]:
response = workflow.invoke(inputs)

[HumanMessage(content='Write a linkedin post on getting a software developer job at IBM under 160 characters', additional_kwargs={}, response_metadata={}, id='4fc43a5a-1ad7-4805-aa69-1763be6431d2'), AIMessage(content='<think>\nOkay, the user wants a LinkedIn post about getting a software developer job at IBM, under 160 characters. Let me start by thinking about the key elements to include. They probably want to highlight their achievement, maybe some skills, and a call to action or gratitude.\n\nFirst, the headline needs to be catchy. Words like "Excited" or "Thrilled" set a positive tone. Mentioning the position and company clearly is essential. Then, maybe include relevant skills or technologies they used, like Java, Python, or cloud computing. IBM has specific projects or methodologies, so mentioning something like IBM Cloud or Agile could add value.\n\nThey might also want to thank mentors, colleagues, or the IBM team. Including a hashtag for visibility, like #SoftwareDeveloper or 

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3-32b` in organization `org_01kpmm0ydye6pt5bm91p1n5w6y` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6053, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
response[1].content